# scene01: all experiments in one notebook (Colab, one T4, ~12 h)

This is the only notebook to run. **Runtime -> Run all**; it checks GitHub first, then runs every experiment until Colab ends the session. Run all again in a later session (any account) and it carries on where it stopped.

**Version 2 (2026-10-10), after the first overnight run.** The first run used 28 frames per recording and decided changes point by point; it could not tell real changes from noise. This version:
- reconstructs **every frame** of both recordings (~370 each at 6 fps) with VGGT in GPU-sized chunks aligned through shared anchor frames;
- fits gravity and the **floor** from horizontal surfaces and estimates **metric scale** from the phone height *and* the sizes of recognised objects (mug, guitar, coaster...);
- segments **every object** with **SAM 2**, merges the masks across views in 3D and **labels every object** with CLIP;
- decides changes **per object** (moved / added / removed / changed in place / shifted), with thresholds from comparing each recording with itself (no labels).

**Where objects are, and how sure (new).** Every object gets a 90% position range: each view's measurement comes with VGGT's own depth confidence and SAM 2's mask wobble, and a Kalman filter fuses the views, so the range narrows with every informative view and with every extra clip. A consistency check (3 = honest) says whether to believe it; the range from views disagreeing is shown alongside.

**Results:** `results/change/RESULTS.md` (one file, by phase, each run titled with what it compares and what it tests) and the interactive viewer (`results/change/viewer.html`, refreshed after every phase).

Per run (`results/change/<RUN_ID>/<step>/`): `objects.png` (every object labelled, border = status), `object_overlays.png`, `heatmaps.png` (uncertainty, deformation, height, reach, approach, interactability), `base_placement.png`, `viewer.html` (3D, labelled objects and heat-map layers), `scene_report.md` (full object inventory, changes with sizes and uncertainty, simulator ranges), `objects.json`, `metrics.json`. `summary_*` steps: changed objects per pair type and how well real changes separate from no-change pairs (AUROC), plus the light vs dark report.

Phases, in order (folder names start with the phase):
1. `1-main` Main comparisons: one or two pairs of every kind (nothing moved; light vs dark; objects moved; moved + lighting), every output, including camera poses sharpened by bundle adjustment `motion_map.png` (how far each object moved and how it deformed, from matched feature points) and `assets/` (each object as a node graph with convex hull and measured strain / relative stiffness, plus contacts). The light before/after pair runs first.
   - `1b-splat` right after that first pair: camera poses sharpened by bundle adjustment, then a Gaussian splat of each of the two recordings (gsplat, started from VGGT, floaters pruned), render quality on frames it never saw, and before vs after rendered from the exact same camera (`splat_compare.png`, `splat_orbit.gif`, `splat_viewer/` web viewer of the real splats, `object_splats/` the changed objects cut out as their own splats).
2. `2-position` Where objects are, and how sure: 3 dark clips of the same scene; 2 light clips before + 2 after.
3. `3-allpairs` Every other pair, so false alarms and real detections can be counted.
4. `4-frames` How many frames / SAM 2 views are enough.
5. `5-robot` Robot variants for the reach and approach maps.
6. `6-repeat` Repeatability: other frame picks, reversed direction.

**Setup:** Runtime -> Change runtime type -> **T4 GPU**. Secrets panel (key icon): `GITHUB_TOKEN` (fine-grained, Contents read/write on UncertainScene) with notebook access on.

In [ ]:
# ---- Config ---------------------------------------------------------------------------
BRANCH = "exp/change/scene01-heatmaps"
RUN_ID = "2026-10-11_scene01-objects-fixed"   # v2 with the labelling and merging fixes; earlier runs stay as they are
VERSION = 2   
PATH_NAME = "change"
EXPECTED_COMMIT = ""                # optional: short sha you expect; setup stops if HEAD differs
RETRY_FAILED = True                 # True re-runs steps that failed with an error (after a code fix)
ONLY_PHASES = None                  # e.g. ["1-main", "2-position"] to restrict; None = the whole queue
SHOW_FIGURES = True                 # show heat maps inline as steps finish
REPO = "FinPThomas/UncertainScene"

## Step 1: GitHub check and setup
Checks the token (found, valid, not expired), that it can see the repo and branch, and does a real test push. If anything is wrong it stops here with what to fix, before any compute is spent.

In [ ]:
# ---- Step 1: GitHub check + setup (every check logged, token never printed) ---------------------------------------
import base64, datetime, os, subprocess, sys, time
from pathlib import Path

SETUP_LOG = []
def say(msg, level="INFO"):
    line = f"{datetime.datetime.now(datetime.timezone.utc):%Y-%m-%d %H:%M:%S}Z {level:5s} {msg}"
    SETUP_LOG.append(line)
    print(line, flush=True)

def fail(msg, hint=""):
    say(msg + (f"\n      fix: {hint}" if hint else ""), "ERROR")
    raise SystemExit(msg)

def on_colab():
    try:
        import google.colab  # noqa: F401
        return True
    except ImportError:
        return False

PLATFORM = "Colab" if on_colab() else "Kaggle" if os.path.isdir("/kaggle/working") else "local"

def secret(name, optional=False):
    """Returns (value, where it came from). Tries Colab secrets, Kaggle secrets, environment."""
    try:
        from google.colab import userdata
        try:
            v = userdata.get(name)
            if v:
                return v, "Colab secrets"
        except Exception as e:  # SecretNotFoundError / NotebookAccessError
            if not optional:
                say(f"Colab secret {name}: {type(e).__name__}", "WARN")
    except ImportError:
        pass
    try:
        from kaggle_secrets import UserSecretsClient
        v = UserSecretsClient().get_secret(name)
        if v:
            return v, "Kaggle secrets"
    except Exception:
        pass
    v = os.environ.get(name)
    return (v, "environment") if v else (None, None)

say(f"[1/8] Python {sys.version.split()[0]}, platform: {PLATFORM}")

# ---- 2. token present and well formed
token, where = secret("GITHUB_TOKEN")
if not token:
    fail("[2/8] GITHUB_TOKEN not found",
         "Colab: key icon in the left bar -> Add new secret named GITHUB_TOKEN, paste the token, switch 'Notebook access' on, then re-run this cell")
token = token.strip()
kind = ("fine-grained (github_pat_)" if token.startswith("github_pat_") else "classic (ghp_)" if token.startswith("ghp_")
        else "unrecognised format")
say(f"[2/8] GITHUB_TOKEN found in {where}: {kind}, {len(token)} characters (value never printed)")
if kind == "unrecognised format":
    say("      the token does not start with github_pat_ or ghp_; check it was pasted whole", "WARN")
os.environ["GITHUB_TOKEN"] = token
label, _ = secret("UNCERTAINSCENE_ACCOUNT", optional=True)  # optional label
if label:
    os.environ["UNCERTAINSCENE_ACCOUNT"] = label
    say(f"      session label from UNCERTAINSCENE_ACCOUNT: {label}")

# ---- 3. token valid, who it belongs to, when it expires
import requests
H = {"Authorization": f"Bearer {token}", "Accept": "application/vnd.github+json", "X-GitHub-Api-Version": "2022-11-28"}
try:
    r = requests.get("https://api.github.com/user", headers=H, timeout=20)
except requests.RequestException as e:
    fail(f"[3/8] cannot reach api.github.com: {type(e).__name__}", "check the runtime has internet (Kaggle: Settings -> Internet on)")
if r.status_code == 401:
    fail("[3/8] GitHub rejected the token (401: invalid, expired or revoked)",
         "github.com/settings/personal-access-tokens/new -> Repository access: Only select repositories -> UncertainScene; "
         "Permissions -> Contents: Read and write -> Generate; then replace the value of the GITHUB_TOKEN secret and re-run")
if r.status_code != 200:
    fail(f"[3/8] GitHub /user returned {r.status_code}: {r.text[:200]}")
exp = r.headers.get("github-authentication-token-expiration", "no expiry date reported")
say(f"[3/8] token valid for GitHub user '{r.json().get('login')}'; expires: {exp}; API calls left this hour: {r.headers.get('x-ratelimit-remaining')}")

# ---- 4. token can see the repository and the branch
r = requests.get(f"https://api.github.com/repos/{REPO}", headers=H, timeout=20)
if r.status_code == 404:
    fail(f"[4/8] token cannot see {REPO} (404)", "fine-grained token: Repository access -> Only select repositories -> add UncertainScene")
if r.status_code != 200:
    fail(f"[4/8] /repos/{REPO} returned {r.status_code}: {r.text[:200]}")
say(f"[4/8] repository {REPO} visible (private: {r.json().get('private')})")
r = requests.get(f"https://api.github.com/repos/{REPO}/branches/{BRANCH}", headers=H, timeout=20)
if r.status_code != 200:
    fail(f"[4/8] branch {BRANCH} not found ({r.status_code})", "check BRANCH in the config cell")
say(f"      branch {BRANCH} at {r.json()['commit']['sha'][:7]}")

# ---- 5. clone (token passed per command, never written to .git/config)
ROOT = Path("/kaggle/working" if PLATFORM == "Kaggle" else "/content")
REPO_DIR = ROOT / "UncertainScene"
auth = base64.b64encode(f"x-access-token:{token}".encode()).decode()
AUTH = ["-c", f"http.https://github.com/.extraheader=AUTHORIZATION: basic {auth}"]

def git(*args, cwd=REPO_DIR):
    p = subprocess.run(["git", *AUTH, *args], cwd=cwd, capture_output=True, text=True)
    if p.returncode:
        raise RuntimeError(p.stderr.replace(token, "***").replace(auth, "***"))
    return p.stdout.strip()

t0 = time.time()
if not (REPO_DIR / ".git").exists():
    git("clone", "--branch", BRANCH, f"https://github.com/{REPO}.git", str(REPO_DIR), cwd=ROOT)
else:
    git("fetch", "origin", BRANCH)
    git("checkout", "-B", BRANCH, f"origin/{BRANCH}")
head = git("rev-parse", "--short", "HEAD")
say(f"[5/8] code at {head} ({git('log', '-1', '--format=%s')}) in {time.time() - t0:.0f} s")
if EXPECTED_COMMIT and not head.startswith(EXPECTED_COMMIT[:7]):
    fail(f"stale code: HEAD is {head}, expected {EXPECTED_COMMIT}", "clear EXPECTED_COMMIT or wait for the push")

# ---- 6. dependencies
t0 = time.time()
pip = [sys.executable, "-m", "pip", "install", "-q"]
for cmd in (pip + ["-e", f"{REPO_DIR}[poc]"], pip + ["--no-deps", "git+https://github.com/facebookresearch/vggt.git"]):
    p = subprocess.run(cmd, capture_output=True, text=True)
    if p.returncode:
        fail(f"[6/8] pip failed: {p.stderr[-800:]}")
# SAM 2 (object masks). Optional: without it the run falls back to point clusters and says so.
p = subprocess.run(pip + ["git+https://github.com/facebookresearch/sam2.git"], capture_output=True, text=True,
                   env={**os.environ, "SAM2_BUILD_CUDA": "0"})
SAM2_OK = p.returncode == 0
if not SAM2_OK:
    say(f"      SAM 2 install failed, objects will come from point clusters: {p.stderr[-300:]}", "WARN")
# YOLOE (whole objects with names, seeds SAM 2 tracks). Optional: without it objects come from SAM 2 alone, as in run 6.
p = subprocess.run(pip + ["ultralytics", "git+https://github.com/ultralytics/CLIP.git"], capture_output=True, text=True)
if p.returncode:
    say(f"      YOLOE (ultralytics) install failed, objects come from SAM 2 alone: {p.stderr[-300:]}", "WARN")
# gsplat (Gaussian splats, phase 1b). Optional: its CUDA code compiles on first use (~10 min, once per session).
p = subprocess.run(pip + ["ninja", "gsplat"], capture_output=True, text=True)
GSPLAT_OK = p.returncode == 0
if p.returncode:
    say(f"      gsplat install failed, the splat step will fail and the rest runs: {p.stderr[-300:]}", "WARN")
sys.path.insert(0, str(REPO_DIR / "src"))
import torch, vggt  # noqa: F401
try:
    import sam2  # noqa: F401
    sam_note = "SAM 2 ok"
except ImportError:
    SAM2_OK, sam_note = False, "SAM 2 missing"
say(f"[6/8] dependencies installed in {time.time() - t0:.0f} s (torch {torch.__version__}, {sam_note})")
# Build gsplat's CUDA code now, while nothing else is in memory. Run 7 lost its session when
# this build started inside the splat step with 10 compiler jobs (RAM 82% -> 97% in 20 s).
# It runs in a separate process with jobs sized to free RAM, retried with fewer if RAM climbs.
if GSPLAT_OK:
    from uncertainscene.recon.gsplat_build import build as build_gsplat
    say("      building gsplat CUDA code (once per session, several minutes)")
    g = build_gsplat()
    if g.get("already_built"):
        say("      gsplat CUDA code already built")
    elif g["ok"]:
        say(f"      gsplat built: {g.get('jobs', 0)} jobs, {g.get('seconds', 0)} s, "
            f"peak RAM {g.get('peak_ram_gb', '?')}/{g.get('ram_total_gb', '?')} GB")
    else:
        say(f"      gsplat build failed, the splat step will retry it: {g['error'][-300:]}", "WARN")

# ---- 7. write access: a dry-run push proves the token can push to this branch
from uncertainscene.infra.publish import Publisher
pub = Publisher(REPO_DIR, BRANCH, token)
pub.ensure_identity()
try:
    pub.check_push_access()
except Exception as e:
    fail(f"[7/8] token cannot push to {BRANCH}: {str(e).replace(token, '***')[:300]}",
         "fine-grained token: Repository permissions -> Contents -> Read and write")
# a real test push (tiny file), so a broken token is caught now, not after hours of compute
test_rel = Path("results") / PATH_NAME / RUN_ID / "logs" / f"github_check_{datetime.datetime.now(datetime.timezone.utc):%Y%m%d_%H%M%S}.txt"
(REPO_DIR / test_rel).parent.mkdir(parents=True, exist_ok=True)
(REPO_DIR / test_rel).write_text("\n".join(SETUP_LOG) + "\n")
try:
    pub.publish([test_rel], f"results({PATH_NAME}): github check {RUN_ID}", fallback_branch=f"results/{RUN_ID}")
except Exception as e:
    fail(f"[7/8] test push failed: {str(e).replace(token, '***')[:300]}",
         "fine-grained token: Repository permissions -> Contents -> Read and write")
say(f"[7/8] GitHub check passed: test file pushed to {BRANCH} ({test_rel})")

## Step 2: GPU and memory guardrails

In [ ]:
# ---- 8. GPU, memory guardrails and run state ---------------------------------------------
import logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(name)s: %(message)s", force=True)
if not torch.cuda.is_available():
    fail("[8/8] no GPU", "Runtime -> Change runtime type -> T4 GPU, then Run all again")
gpu_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3

from uncertainscene.infra import Run
from uncertainscene.infra.guard import ResourceLimit, read_usage
from uncertainscene.infra.store import Store, repo_slug
from uncertainscene.recon.vggt import max_frames_for_gpu

run = Run.from_env(run_id=RUN_ID, path=PATH_NAME, branch=BRANCH)
b = run.budget
say(f"[8/8] GPU {torch.cuda.get_device_name(0)} {gpu_gb:.1f} GB: fits ~{max_frames_for_gpu()} frames per reconstruction "
    "(more are capped; CUDA out-of-memory retries with 20% fewer)")
say(f"      machine now: {read_usage(b.work_dir).line()}")
say(f"      guardrails: RAM warn {b.ram_soft_frac:.0%} / stop {b.ram_hard_frac:.0%}, disk warn {b.disk_soft_free_gb} GB / "
    f"stop {b.disk_hard_free_gb} GB free, checked every {b.monitor_interval_s} s and between every stage of a step; "
    f"a step only starts if its RAM, disk and time fit; session plan {b.session_hours} h minus {b.safety_minutes} min")
say(f"      session label {run.account}; run state:\n" + run.state.summary())

CACHE = Path(b.work_dir) / "cache"      # reconstructions reused across steps (outside step scratch)
RUN_DIR = REPO_DIR / "results" / PATH_NAME / RUN_ID
try:
    store = Store(repo_slug(REPO_DIR))            # dense clouds as release assets (best effort)
except Exception as e:
    store = None
    say(f"no release store ({type(e).__name__}); dense clouds stay local", "WARN")

# push the setup log so a failed or odd session can be diagnosed from GitHub
log_rel = Path("results") / PATH_NAME / RUN_ID / "logs" / f"setup_{run.account}.txt"
(REPO_DIR / log_rel).parent.mkdir(parents=True, exist_ok=True)
(REPO_DIR / log_rel).write_text("\n".join(SETUP_LOG) + "\n")
try:
    pub.publish([log_rel], f"results({PATH_NAME}): setup log {RUN_ID} ({run.account})", fallback_branch=f"results/{RUN_ID}")
    say(f"setup log pushed: {log_rel}")
except Exception as e:
    say(f"could not push the setup log: {str(e).replace(token, '***')[:200]}", "WARN")

## Step 3: all experiments (runs until the session ends)

In [ ]:
# ---- The queue: runs until the session ends ---------------------------------------------
# 1-main the six main pairs (light before/after first, then 1b-splat: Gaussian splats of that pair) -> 2-position where objects are, with 1-3 clips
# -> 3-allpairs every other pair -> 4-frames how many frames are enough -> 5-robot robot variants
# -> 6-repeat repeatability. Each step: skipped if done (by any account) or claimed by a live
# session elsewhere; not started if the RAM, disk or session time it needs is not available.
# Every step pushes its own step_log.txt with stage timings and memory use.
import gc
from IPython.display import Image, Markdown, display
from uncertainscene.eval import summarize
from uncertainscene.experiments.poc import trim_cache
from uncertainscene.experiments.scene01 import queue, step_fn

from uncertainscene.eval import collate
from uncertainscene.viz import site

def summary_step(out, work):
    summarize.write(RUN_DIR, out)
    # the one results file and the viewer, refreshed after every phase (both committed with this step)
    res_md = collate.write(RUN_DIR.parent)
    site.build([RUN_DIR], RUN_DIR.parent / "viewer.html", summary_md=res_md.read_text(), max_mb=4.8)
    try:
        pub.publish([res_md.relative_to(REPO_DIR), (RUN_DIR.parent / "viewer.html").relative_to(REPO_DIR)],
                    f"results({PATH_NAME}): RESULTS.md and viewer after {out.name}", fallback_branch=f"results/{RUN_ID}")
    except Exception as e:
        say(f"could not push RESULTS.md / viewer.html: {str(e).replace(token, '***')[:200]}", "WARN")
    return f"summary of {len(summarize.load(RUN_DIR))} runs; RESULTS.md and viewer.html refreshed"

counts, t_start = {}, time.time()
try:
    for item in queue():
        name = item if isinstance(item, str) else item.name
        if ONLY_PHASES and name.split("_")[0] not in ONLY_PHASES:
            continue
        t0 = time.time()
        if isinstance(item, str):
            status = run.step(item, summary_step, needs=dict(disk_gb=0.2, ram_gb=1, minutes=2), retry_failed=RETRY_FAILED)
        else:
            status = run.step(item.name, step_fn(item, CACHE, store, run.guard.check), needs=item.needs,
                              retry_failed=RETRY_FAILED)
            trim_cache(CACHE, max_gb=8)
            gc.collect()
            torch.cuda.empty_cache()
        counts[status] = counts.get(status, 0) + 1
        if status not in ("skipped",):
            say(f"{name}: {status} in {(time.time() - t0) / 60:.1f} min | {run.state.step(name).note or ''} | "
                f"{read_usage(b.work_dir).line()} | {run.guard.minutes_left():.0f} min left")
        if status == "failed":
            say(f"      see results/{PATH_NAME}/{RUN_ID}/{name}/error.txt and step_log.txt", "WARN")
        if SHOW_FIGURES and status == "done" and not isinstance(item, str) and item.outputs == "full":
            display(Markdown(f"### {name}\n{run.state.step(name).note}"))
            for fig in ("splat_compare.png", "objects.png", "position_map.png", "position_closeups.png", "heatmaps.png"):
                if (run.out_dir(name) / fig).exists():
                    display(Image(str(run.out_dir(name) / fig), width=1100))
except ResourceLimit as e:
    say(f"stopping cleanly, everything finished so far is pushed: {e}", "WARN")
say(f"session: {counts} in {(time.time() - t_start) / 3600:.1f} h")

## Step 4: final summary

In [ ]:
# ---- Finish: refresh the overall summary and show the headline ---------------------------
try:
    run.step(f"summary_session_{run.account}", summary_step, needs=dict(disk_gb=0.2, ram_gb=1, minutes=2))
except ResourceLimit as e:
    print("No time left for the final summary:", e)
run.close()
print(run.state.summary())
s = run.out_dir(f"summary_session_{run.account}") / "summary.md"
if s.exists():
    display(Markdown(s.read_text()[:6000]))